# Customer revenue pipeline — preprocessing

This notebook creates a synthetic analyst dataset, repairs missing values, engineers features, and publishes train/test splits for downstream model notebooks.

In [ ]:
# @node: Load customer data  [input]  out=raw_df
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 520
channels = np.array(["search", "social", "email", "partner"])
regions = np.array(["north", "south", "west"])
raw_df = pd.DataFrame(
    {
        "date": pd.date_range("2026-01-01", periods=n, freq="D"),
        "channel": rng.choice(channels, n, p=[0.36, 0.28, 0.22, 0.14]),
        "region": rng.choice(regions, n),
        "visitors": rng.integers(500, 5200, n),
        "ad_spend": rng.gamma(shape=5.5, scale=260.0, size=n).round(2),
        "discount": rng.choice([0.0, 0.05, 0.10, 0.15], n, p=[0.48, 0.22, 0.20, 0.10]),
        "competitor_index": rng.normal(1.0, 0.12, n).round(3),
    }
)
channel_effect = raw_df["channel"].map({"search": 920, "social": 680, "email": 1130, "partner": 760})
region_effect = raw_df["region"].map({"north": 120, "south": -90, "west": 40})
noise = rng.normal(0, 1300, n)
raw_df["revenue"] = (
    850
    + raw_df["visitors"] * 2.85
    + raw_df["ad_spend"] * 1.72
    + raw_df["discount"] * 14500
    - raw_df["competitor_index"] * 1900
    + channel_effect
    + region_effect
    + noise
).round(2)
raw_df.loc[rng.choice(n, 22, replace=False), "ad_spend"] = np.nan
display(raw_df.head())

In [ ]:
# @node: Clean features  [transform]  in=raw_df<-Load customer data.raw_df  out=feature_df
import pandas as pd

feature_df = raw_df.copy()
feature_df["ad_spend"] = feature_df["ad_spend"].fillna(
    feature_df.groupby("channel")["ad_spend"].transform("median")
)
feature_df["discount"] = feature_df["discount"].fillna(0.0)
feature_df["is_weekend"] = (feature_df["date"].dt.dayofweek >= 5).astype(float)
feature_df["month"] = feature_df["date"].dt.month.astype(float)
feature_df["cost_per_visitor"] = feature_df["ad_spend"] / feature_df["visitors"]
encoded = pd.get_dummies(feature_df[["channel", "region"]], dtype=float)
feature_df = pd.concat([feature_df, encoded], axis=1)
feature_df = feature_df.sort_values("date").reset_index(drop=True)
print(f"prepared {len(feature_df)} rows with {len(encoded.columns)} categorical indicators")

In [ ]:
# @node: Train test split  [transform]  in=feature_df<-Clean features.feature_df  out=train_df,test_df,feature_cols
base_cols = ["visitors", "ad_spend", "discount", "competitor_index", "is_weekend", "month", "cost_per_visitor"]
indicator_cols = [c for c in feature_df.columns if c.startswith("channel_") or c.startswith("region_")]
feature_cols = base_cols + indicator_cols
cutoff = int(len(feature_df) * 0.75)
train_df = feature_df.iloc[:cutoff].copy()
test_df = feature_df.iloc[cutoff:].copy()
print(f"train rows: {len(train_df)} | test rows: {len(test_df)} | features: {len(feature_cols)}")